# Ablation: `web_search` availability vs. eval rollup

[Issue #14](https://github.com/davidzornek/adk/issues/14) — a comparative eval write-up, not new
library code. It reuses the hand-written eval cases and metrics from
[`eval_plan_then_act_demo.ipynb`](../demos/eval_plan_then_act_demo.ipynb) and runs them through
`adk.demos.plan_then_act_demo.DemoPlanThenActAgent` in two configurations, named for exactly
what differs between them:

1. **`web_search` enabled** — the agent as-is: `web_search` and `calculate` both live.
2. **`web_search` disabled** — same agent, same planner prompt, same config; only the
   `web_search` tool implementation is swapped for one that always fails. `calculate` is
   untouched.

The question this is evidence for: does the eval harness's `task_success` /
`plan_execution_alignment` split actually distinguish *"the plan was right but a tool failed"*
from *"the plan was wrong"*? Disabling one tool while leaving the planner's routing logic
untouched is a clean way to check that, and it doubles as a small case study in
`DegradedModeExecutor`'s fault boundary.

## Results at a glance

| metric | `web_search` enabled | `web_search` disabled |
|---|---|---|
| pass rate | 1.00 | 0.33 |
| alignment rate | 1.00 | 1.00 |
| avg. steps to completion | 1.67 | 1.67 |

| case | needs `web_search`? | success, enabled → disabled |
|---|---|---|
| `population_lookup` | yes | True → False |
| `arithmetic` | no (control) | True → True |
| `combined_population` | yes | True → False |

**Disabling `web_search` drops pass rate on the two cases that depend on it, while alignment
rate and the `arithmetic` control case don't move at all** — the planner keeps routing steps to
the right executor/tool either way; only the tool call itself fails. That's the harness
correctly separating *"the plan was right but a dependency failed"* from *"the plan was
wrong"*. Full numbers and per-case detail are below; see "Honest read of the results" at the
end for the caveats (3 hand-written cases, single run each — directional, not statistically
rigorous).

## Setup

Same two keys as the other demo notebooks:

- `ANTHROPIC_API_KEY` — https://console.anthropic.com/
- `TAVILY_API_KEY` — https://tavily.com/

Copy `.env.example` (repo root) to `.env` and paste your keys in there — `load_dotenv()` below
loads it into this process.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

for key in ("ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'MISSING'}")


ANTHROPIC_API_KEY: set
TAVILY_API_KEY: set


## The cases

The same three hand-written cases as `eval_plan_then_act_demo.ipynb` (`docs/demos/`), so the
`web_search`-enabled rollup below is directly comparable to that notebook's:

- **`population_lookup`** — pure lookup, should route to `search` only. Depends on `web_search`.
- **`arithmetic`** — pure arithmetic, no lookup needed, should route to `calc` only. Does not
  touch `web_search` — this is the control case.
- **`combined_population`** — two `search` steps (France, Germany) then one `calc` step. Note
  that in this pattern `calc`'s `tool_args` are fixed by the planner *before* any tool runs (see
  `plan_then_act_demo`'s module docstring), so the calc step never actually consumes the search
  results — it degrades or succeeds independently of them.

In [2]:
from adk.eval_harness.cases import EvalCase, ExpectedStep
from adk.eval_harness.local_harness import rollup, run_and_score
from adk.eval_harness.metrics import (
    latency,
    plan_execution_alignment,
    steps_to_completion,
    task_success,
    token_counts,
)

CASES = [
    EvalCase(
        id="population_lookup",
        task="What is the current population of France?",
        expected_steps=[ExpectedStep(executor_id="search", tool_name="web_search")],
    ),
    EvalCase(
        id="arithmetic",
        task="What is 482 times 17?",
        expected_steps=[ExpectedStep(executor_id="calc", tool_name="calculate")],
    ),
    EvalCase(
        id="combined_population",
        task=(
            "Look up the current population of France and Germany, then calculate their "
            "combined population."
        ),
        expected_steps=[
            ExpectedStep(executor_id="search", tool_name="web_search"),
            ExpectedStep(executor_id="search", tool_name="web_search"),
            ExpectedStep(executor_id="calc", tool_name="calculate"),
        ],
    ),
]

METRICS = [task_success, steps_to_completion, plan_execution_alignment, latency, token_counts]


## Two agent configurations

`DemoPlanThenActAgent` (the `web_search`-enabled configuration) is imported unmodified from
`adk.demos.plan_then_act_demo`.

`WebSearchDisabledAgent` is assembled from the exact same building blocks
`DemoPlanThenActAgent._build_graph` uses (`build_plan_then_act_planner`, `AnthropicRunnable`,
`DegradedModeExecutor`, `build_plan_then_act_graph`) — same planner system prompt, same config,
same `calculate` tool. The *only* change is what the `search` executor's `web_search` tool does:
instead of a live Tavily call, it always raises. The planner still knows about and plans for
`web_search` exactly as before; `DegradedModeExecutor`'s own fault boundary is what turns that
into a `"degraded"` step, the same way a real Tavily outage would. This is what makes it a
disabled *tool*, not a change to the *planner's routing logic* — routing should be unaffected,
and only `task_success` on search-dependent cases should move.

`variant` (the `@property` below) isn't specific to this notebook — every
`PlannerExecutorBase` subclass must set it, as a stable id that eval/trace metadata
(`invoke_trace`) uses to label which run came from which agent. It's set to
`"plan_then_act_demo_web_search_disabled"` here so runs from this notebook are distinguishable
from the baseline demo's own `"plan_then_act_demo"` in any shared traces — naming it after what
changed, the same convention as everything else in this notebook.

In [3]:
from typing import Any

from anthropic import Anthropic

from adk.anthropic_client import get_anthropic_client
from adk.anthropic_runnables import AnthropicRunnable
from adk.demos.plan_then_act_demo import (
    DEFAULT_MODEL,
    DRAFTER_SYSTEM_PROMPT,
    PLANNER_SYSTEM_PROMPT,
    DemoPlanThenActAgent,
    calculate,
    default_config,
)
from adk.planner_executor.base import PlannerExecutorBase
from adk.planner_executor.executor import DegradedModeExecutor
from adk.planner_executor.graph import build_plan_then_act_graph
from adk.planner_executor.planner import build_plan_then_act_planner


def disabled_web_search(args: dict[str, Any], ctx: dict[str, Any]) -> dict[str, Any]:
    """Ablation stand-in for ``web_search``: always fails, as if the provider were down."""
    raise RuntimeError("web_search is disabled for this ablation run")


class WebSearchDisabledAgent(PlannerExecutorBase):
    """``DemoPlanThenActAgent`` with ``web_search`` swapped for a tool that always fails."""

    def __init__(self, *, client: Anthropic | None = None, model: str = DEFAULT_MODEL) -> None:
        self._client = client or get_anthropic_client()
        self._model = model
        super().__init__(config=default_config())

    @property
    def variant(self) -> str:
        return "plan_then_act_demo_web_search_disabled"

    def _input_to_state(self, input: dict[str, Any]) -> dict[str, Any]:
        return {"task": input["task"], "context": input.get("context", {})}

    def _build_graph(self) -> Any:
        planner = build_plan_then_act_planner(
            self._client, model=self._model, system_prompt=PLANNER_SYSTEM_PROMPT,
        )
        drafter = AnthropicRunnable(
            self._client, model=self._model, system_prompt=DRAFTER_SYSTEM_PROMPT,
        )
        executors = {
            "search": DegradedModeExecutor(
                executor_id="search",
                tools={"web_search": disabled_web_search},
                degraded_mode="Web search is temporarily unavailable.",
            ),
            "calc": DegradedModeExecutor(
                executor_id="calc",
                tools={"calculate": calculate},
                degraded_mode="Calculation failed.",
            ),
        }
        return build_plan_then_act_graph(
            planner=planner,
            executors=executors,
            drafter=drafter,
            drafter_system=DRAFTER_SYSTEM_PROMPT,
            validator=None,
            config=self.config,
        )


web_search_enabled_agent = DemoPlanThenActAgent()
web_search_disabled_agent = WebSearchDisabledAgent()


## Running both configurations

Six live Anthropic + Tavily round trips total (three cases x two configurations) — this cell
takes a little while. `TAVILY_API_KEY` is still required even for the `web_search`-disabled run:
the search executor's tool is swapped for `disabled_web_search`, but Tavily itself is never
called in that configuration — the key is only exercised by the `web_search`-enabled run.

In [4]:
web_search_enabled_rows = run_and_score(CASES, web_search_enabled_agent, METRICS)
web_search_disabled_rows = run_and_score(CASES, web_search_disabled_agent, METRICS)


executor search degraded: web_search is disabled for this ablation run


executor search degraded: web_search is disabled for this ablation run


executor search degraded: web_search is disabled for this ablation run


## Scored rows, side by side

In [5]:
def print_rows(label: str, rows: list[dict]) -> None:
    print(f"--- {label} ---")
    for row in rows:
        print(f"{row['case_id']!r}:")
        for key, value in row.items():
            if key == "case_id":
                continue
            print(f"  {key}: {value}")


print_rows("web_search enabled", web_search_enabled_rows)
print()
print_rows("web_search disabled", web_search_disabled_rows)


--- web_search enabled ---
'population_lookup':
  run_id: d7c1483a-0c46-4cca-bc5d-540c969582fd
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 6550.307833007537
  tokens_in: 2905
  tokens_out: 295
  tokens_total: 3200
'arithmetic':
  run_id: 6d3875e4-cb7f-44a7-af5f-28412c04c358
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 3073.023750039283
  tokens_in: 1432
  tokens_out: 88
  tokens_total: 1520
'combined_population':
  run_id: 946ad1ba-00a4-48af-8ea6-172c1621a707
  success: True
  n_steps: 3
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 3
  aligned: True
  n_expected_steps: 3
  n_executed_steps: 3
  mismatches: []
  latency_ms: 6699.4245829992
  tokens_in: 5015
  toke

## Rollups and the diff

`rollup()` is the same function used in `eval_plan_then_act_demo.ipynb`. The diff below is the
"small script" the issue calls for — no new library code, just a dict subtraction over whatever
keys both rollups share. These are the numbers summarized in "Results at a glance" above.

In [6]:
web_search_enabled_rollup = rollup(web_search_enabled_rows)
web_search_disabled_rollup = rollup(web_search_disabled_rows)

print("web_search enabled: ", web_search_enabled_rollup)
print("web_search disabled:", web_search_disabled_rollup)


def diff_rollups(before: dict, after: dict) -> dict:
    """Per-key delta (after - before) over keys present in both rollups."""
    return {
        key: after[key] - before[key]
        for key in before
        if key in after and isinstance(before[key], (int, float))
    }


diff_rollups(web_search_enabled_rollup, web_search_disabled_rollup)


web_search enabled:  {'n_cases': 3, 'pass_rate': 1.0, 'avg_steps_to_completion': 1.6666666666666667, 'alignment_rate': 1.0}
web_search disabled: {'n_cases': 3, 'pass_rate': 0.3333333333333333, 'avg_steps_to_completion': 1.6666666666666667, 'alignment_rate': 1.0}


{'n_cases': 0,
 'pass_rate': -0.6666666666666667,
 'avg_steps_to_completion': 0.0,
 'alignment_rate': 0.0}

## Per-case read: which cases moved, and on which metric

`task_success` (did the run finish with zero degraded steps) is expected to flip on the two
search-dependent cases and stay put on `arithmetic`. `plan_execution_alignment` (did the planner
route to the right executor/tool, independent of whether the tool call itself succeeded) is
expected to stay aligned on all three — the planner never sees which configuration it's running
in, only `DegradedModeExecutor` does.

In [7]:
by_case = {
    "web_search enabled": {r["case_id"]: r for r in web_search_enabled_rows},
    "web_search disabled": {r["case_id"]: r for r in web_search_disabled_rows},
}

for case in CASES:
    b = by_case["web_search enabled"][case.id]
    a = by_case["web_search disabled"][case.id]
    print(
        f"{case.id:20s} success: {str(b['success']):5s} -> {str(a['success']):5s}   "
        f"aligned: {str(b['aligned']):5s} -> {str(a['aligned']):5s}   "
        f"degraded_steps: {b['n_degraded_steps']} -> {a['n_degraded_steps']}",
    )


population_lookup    success: True  -> False   aligned: True  -> True    degraded_steps: 0 -> 1
arithmetic           success: True  -> True    aligned: True  -> True    degraded_steps: 0 -> 0
combined_population  success: True  -> False   aligned: True  -> True    degraded_steps: 0 -> 2


## Honest read of the results

**Directional, small-N — 3 hand-written cases, single run each, no repeats or confidence
intervals. This is not a statistically rigorous claim about the agent's reliability under tool
outages; it's a worked example of what one such outage looks like through this harness.**

What the numbers above show, and why:

- **`arithmetic` is the control.** It doesn't touch `web_search`, so both configurations score
  it identically. If it had moved too, that would be a sign the ablation leaked somewhere it
  shouldn't have (e.g. the planner's prompt or the `calc` tool itself), not a real ablation
  effect.
- **`population_lookup` and `combined_population` are the treatment.** Both depend on
  `web_search`; both flip `task_success` to `False` when it's disabled, surfaced via
  `n_degraded_steps > 0` — `DegradedModeExecutor` catching the forced `RuntimeError` and turning
  it into a `"degraded"` step rather than crashing the run.
- **`plan_execution_alignment` is the metric that does *not* move.** The planner's system prompt
  and available `output_plan` schema are identical in both configurations — only the tool
  implementation the executor calls is different — so a correctly-routed step (`executor_id`
  and `tool` matching `expected_steps`) stays correctly routed even when its execution degrades.
  Had alignment dropped too, that would suggest the planner itself was reacting to the outage
  (e.g. skipping the search step entirely, or substituting `calculate`), which is a materially
  different — and worse — failure mode than a clean tool degradation.
- **This is why the harness carries both metrics rather than one.** A single pass/fail number
  can't distinguish "the plan was right and a dependency failed" from "the plan was wrong" —
  those call for different fixes (retry/fallback infrastructure vs. prompt or routing changes).
  Seeing `success=False` alongside `aligned=True` on the two `web_search`-dependent cases is
  exactly that distinction made visible in one row.